In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
data = pd.read_excel('../artifacts/SrilankanCommonFoods.xlsx')

In [4]:
data.head()

,Food,Quantity,Calories (kcal),Carbohydrate (g),Protein (g),Fat (g)
0,White Rice,80g,110 kcal,24g,2g,0g
1,Brown Rice,80g,90 kcal,19g,2g,1g
2,Red Rice,80g,112 kcal,23g,2.4g,0.8g
3,White Bread,100g,270 kcal,50g,9g,3g
4,Pasta,250g,393 kcal,79g,14g,1.7g


In [5]:
##Data Preprocessing

In [7]:
data.shape

(119, 6)

In [10]:
data.duplicated().sum()

np.int64(0)

In [11]:
data.isnull().sum()

Food                0
Quantity            0
Calories (kcal)     0
Carbohydrate (g)    0
Protein (g)         0
Fat (g)             0
dtype: int64

In [17]:
data['Calories (kcal)'] = data['Calories (kcal)'].str.replace('kcal','').astype(float)

In [19]:
data['Carbohydrate (g)'] = data['Carbohydrate (g)'].str.replace('g','').astype(float)
data['Protein (g)'] = data['Protein (g)'].str.replace('g','').astype(float)
data['Fat (g)'] = data['Fat (g)'].str.replace('g','').astype(float)

In [21]:
data.drop(columns=['Calories', 'Carbohydrate', 'Protein', 'Fat'], inplace=True)

In [22]:
data.head()

,Food,Quantity,Calories (kcal),Carbohydrate (g),Protein (g),Fat (g)
0,White Rice,80g,110.0,24.0,2.0,0.0
1,Brown Rice,80g,90.0,19.0,2.0,1.0
2,Red Rice,80g,112.0,23.0,2.4,0.8
3,White Bread,100g,270.0,50.0,9.0,3.0
4,Pasta,250g,393.0,79.0,14.0,1.7


In [37]:
#step 2
import re

In [26]:
# 1. Extract serving size (Quantity column → numeric grams)
data['Quantity_g'] = data['Quantity'].str.extract(r'(\d+\.?\d*)').astype(float)

In [30]:
# 2. Normalize nutrition values (per 100g)
data['Calories (kcal)'] = (data['Calories (kcal)'] / data['Quantity_g']) * 100
data['Carbohydrate (g)'] = (data['Carbohydrate (g)'] / data['Quantity_g']) * 100
data['Protein (g)'] = (data['Protein (g)'] / data['Quantity_g']) * 100
data['Fat (g)'] = (data['Fat (g)'] / data['Quantity_g']) * 100

In [31]:
# 3. Round for clean output
data[['Calories (kcal)', 'Carbohydrate (g)', 'Protein (g)', 'Fat (g)']] = \
    data[['Calories (kcal)', 'Carbohydrate (g)', 'Protein (g)', 'Fat (g)']].round(2)

In [36]:
data.head()

,Food,Quantity,Calories (kcal),Carbohydrate (g),Protein (g),Fat (g)
0,White Rice,100g,171.88,37.50,3.12,0.00
1,Brown Rice,100g,140.62,29.69,3.12,1.56
2,Red Rice,100g,175.00,35.94,3.75,1.25
3,White Bread,100g,270.00,50.00,9.00,3.00
4,Pasta,100g,62.88,12.64,2.24,0.27


In [33]:
# Change Quantity column to always show "100g"
data['Quantity'] = '100g'

In [35]:
data.drop(columns=['Quantity_g'], inplace=True)

In [38]:
#step 3:Add Derived Nutrition Features

In [39]:
# 1. High/Low Feature Flags
data['High_Carb'] = data['Carbohydrate (g)'] > 30      # >30g per 100g = high-carb
data['High_Protein'] = data['Protein (g)'] > 10        # >10g per 100g = high-protein
data['High_Fat'] = data['Fat (g)'] > 10                # >10g per 100g = high-fat
data['Low_Fat'] = data['Fat (g)'] < 3                  # <3g per 100g = low-fat
data['Low_Protein'] = data['Protein (g)'] < 5

In [41]:
# 2. Vitamin K-rich foods (Warfarin interaction)
vitamin_k_foods = ['Spinach', 'Kale', 'Broccoli', 'Cabbage', 'Green Peas',
                   'Lettuce', 'Collard Greens', 'Swiss Chard', 'Mustard Greens']

data['Vitamin_K_Rich'] = data['Food'].isin(vitamin_k_foods)

In [42]:
# 3. Energy Density (kcal per gram)
data['Energy_Density'] = data['Calories (kcal)'] / 100  

In [43]:
# 4. Carb-to-Protein Ratio
data['Carb_to_Protein_Ratio'] = data['Carbohydrate (g)'] / data['Protein (g)']

In [44]:
# 5. Fiber Estimation (simple model — can improve later)
# Average fiber % by carb amount (approximation)
data['Estimated_Fiber (g)'] = data['Carbohydrate (g)'] * 0.1   # assume 10% of carbs is fiber

In [45]:
data.head()

,Food,Quantity,Calories (kcal),Carbohydrate (g),Protein (g),Fat (g),High_Carb,High_Protein,High_Fat,Low_Fat,Low_Protein,Vitamin_K_Rich,Energy_Density,Carb_to_Protein_Ratio,Estimated_Fiber (g)
0,White Rice,100g,171.88,37.50,3.12,0.00,True,False,False,True,True,False,1.7188,12.019231,3.750
1,Brown Rice,100g,140.62,29.69,3.12,1.56,False,False,False,True,True,False,1.4062,9.516026,2.969
2,Red Rice,100g,175.00,35.94,3.75,1.25,True,False,False,True,True,False,1.7500,9.584000,3.594
3,White Bread,100g,270.00,50.00,9.00,3.00,True,False,False,False,False,False,2.7000,5.555556,5.000
4,Pasta,100g,62.88,12.64,2.24,0.27,False,False,False,True,True,False,0.6288,5.642857,1.264
